<a href="https://colab.research.google.com/github/Joshitha19/Samridhi/blob/main/samridhi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest as SklearnIsolationForest

# =====================================================================
# 1. CUSTOM ISOLATION FOREST IMPLEMENTATION
# =====================================================================

class IsolationTreeNode:
    def __init__(self, left=None, right=None, split_feature=None, split_value=None, size=0):
        self.left = left
        self.right = right
        self.split_feature = split_feature
        self.split_value = split_value
        self.size = size

def c(n):
    """
    Average path length of unsuccessful searches in a Binary Search Tree (BST).
    Equivalent to the normalization factor in Isolation Forest.
    """
    if n <= 1:
        return 0
    if n == 2:
        return 1
    euler_gamma = 0.5772156649
    return 2 * (math.log(n - 1) + euler_gamma) - 2 * (n - 1) / n

class CustomIsolationTree:
    def __init__(self, rng):
        self.root = None
        self.rng = rng

    def fit(self, data, current_height, max_height):
        n_samples = len(data)
        if n_samples <= 1 or current_height >= max_height:
            return IsolationTreeNode(size=n_samples)

        # Choose random feature
        features = list(data.columns)
        split_feature = self.rng.choice(features)

        # Find min and max
        values = data[split_feature].values
        min_val = values.min()
        max_val = values.max()

        if min_val == max_val:
            return IsolationTreeNode(size=n_samples)

        # Choose random split value
        split_value = min_val + self.rng.random() * (max_val - min_val)

        left_data = data[data[split_feature] < split_value]
        right_data = data[data[split_feature] >= split_value]

        return IsolationTreeNode(
            left=self.fit(left_data, current_height + 1, max_height),
            right=self.fit(right_data, current_height + 1, max_height),
            split_feature=split_feature,
            split_value=split_value,
            size=n_samples
        )

class CustomIsolationForest:
    def __init__(self, num_trees=15, sub_sample_size=256, seed=1337):
        self.num_trees = num_trees
        self.sub_sample_size = sub_sample_size
        self.rng = np.random.default_rng(seed)
        self.trees = []

    def fit(self, data):
        self.trees = []
        n_samples = len(data)
        if n_samples == 0:
            return

        size = min(self.sub_sample_size, n_samples)
        max_height = math.ceil(math.log2(size))

        for _ in range(self.num_trees):
            # Select random subsample indices without replacement
            indices = self.rng.choice(n_samples, size=size, replace=False)
            sample = data.iloc[indices].copy()

            tree = CustomIsolationTree(self.rng)
            tree.root = tree.fit(sample, 0, max_height)
            self.trees.append(tree)

    def _path_length(self, x, node, current_depth):
        if node is None or node.size <= 1:
            return current_depth + c(node.size if node else 0)

        if node.split_feature is None:
            return current_depth + c(node.size)

        if x[node.split_feature] < node.split_value:
            return self._path_length(x, node.left, current_depth + 1)
        else:
            return self._path_length(x, node.right, current_depth + 1)

    def score(self, x):
        if not self.trees:
            return 0.5

        sum_path_length = 0
        for tree in self.trees:
            sum_path_length += self._path_length(x, tree.root, 0)

        avg_path_length = sum_path_length / len(self.trees)
        n = min(self.sub_sample_size, self.trees[0].root.size if self.trees[0].root else 0)
        avg_c = c(n)

        if avg_c == 0:
            return 0.5
        return math.pow(2, -avg_path_length / avg_c)


# =====================================================================
# 2. COLAB TESTING SUITE
# =====================================================================

if __name__ == "__main__":
    print("----------------------------------------------------------------------")
    print("      Samridhi: Isolation Forest Anomaly Detection (Colab Test)      ")
    print("----------------------------------------------------------------------")

    # Generate synthetic transaction dataset
    np.random.seed(42)
    n_records = 200

    amounts = np.random.exponential(scale=1500, size=n_records) # normal transactions
    directions = np.random.choice([1, -1], size=n_records, p=[0.3, 0.7])
    categories = np.random.choice([2, 3, 4, 5, 7, 8], size=n_records) # Category Indexes
    days = np.random.randint(0, 7, size=n_records)

    # Force insert salary credit (Consistent)
    amounts[0] = 45000
    directions[0] = 1
    categories[0] = 1 # Income
    days[0] = 1 # Monday

    # Inject anomaly transaction (Stress-Test: high volume outflow outlier)
    amounts[50] = 120000
    directions[50] = -1 # Outflow
    categories[50] = 8 # Other
    days[50] = 0 # Sunday

    # Inject anomaly transaction (Stress-Test: high volume shopping outflow)
    amounts[120] = 95000
    directions[120] = -1
    categories[120] = 4 # Shopping
    days[120] = 3

    # Build Pandas DataFrame
    df = pd.DataFrame({
        'amount': amounts,
        'type': directions,
        'category': categories,
        'dayOfWeek': days
    })

    print(f"Dataset generated. Rows: {df.shape[0]}, Features: {list(df.columns)}")

    # 1. Train Custom Model
    print("\n--- TRAINING CUSTOM ISOLATION FOREST ---")
    custom_forest = CustomIsolationForest(num_trees=15, sub_sample_size=256, seed=1337)
    custom_forest.fit(df)

    # Calculate scores on the samples
    custom_scores = [custom_forest.score(row) for _, row in df.iterrows()]
    df['custom_anomaly_score'] = custom_scores
    print("Custom model trained successfully.")

    # 2. Train Scikit-Learn Model
    print("\n--- TRAINING SCIKIT-LEARN ISOLATION FOREST ---")
    clf = SklearnIsolationForest(n_estimators=15, max_samples=min(256, n_records), random_state=1337)
    clf.fit(df[['amount', 'type', 'category', 'dayOfWeek']])
    sklearn_offset = clf.decision_function(df[['amount', 'type', 'category', 'dayOfWeek']])
    df['sklearn_anomaly_score'] = 0.5 - sklearn_offset

    # Print Results for Injected Anomalies
    print("\n--- TESTING OUTLIERS ---")

    # Normal Inflow (Salary credit)
    print(f"\n[Record 0 - Normal Salary Inflow]:")
    print(f"-> Custom Forest Anomaly Score: {df.loc[0, 'custom_anomaly_score']:.4f} (Flagged Anomaly: {df.loc[0, 'custom_anomaly_score'] > 0.58})")
    print(f"-> Sklearn Forest Anomaly Score: {df.loc[0, 'sklearn_anomaly_score']:.4f}")

    # Outflow Anomaly (Outflow outlier)
    print(f"\n[Record 50 - High Volume Outflow Anomaly]:")
    print(f"-> Custom Forest Anomaly Score: {df.loc[50, 'custom_anomaly_score']:.4f} (Flagged Anomaly: {df.loc[50, 'custom_anomaly_score'] > 0.58})")
    print(f"-> Sklearn Forest Anomaly Score: {df.loc[50, 'sklearn_anomaly_score']:.4f}")

    print("\n--- TOP 3 OUTLIERS (SORTED BY SCORE) ---")
    print(df.sort_values(by='custom_anomaly_score', ascending=False).head(3))

----------------------------------------------------------------------
      Samridhi: Isolation Forest Anomaly Detection (Colab Test)      
----------------------------------------------------------------------
Dataset generated. Rows: 200, Features: ['amount', 'type', 'category', 'dayOfWeek']

--- TRAINING CUSTOM ISOLATION FOREST ---
Custom model trained successfully.

--- TRAINING SCIKIT-LEARN ISOLATION FOREST ---

--- TESTING OUTLIERS ---

[Record 0 - Normal Salary Inflow]:
-> Custom Forest Anomaly Score: 0.6899 (Flagged Anomaly: True)
-> Sklearn Forest Anomaly Score: 0.7525

[Record 50 - High Volume Outflow Anomaly]:
-> Custom Forest Anomaly Score: 0.6695 (Flagged Anomaly: True)
-> Sklearn Forest Anomaly Score: 0.8156

--- TOP 3 OUTLIERS (SORTED BY SCORE) ---
       amount  type  category  dayOfWeek  custom_anomaly_score  \
0     45000.0     1         1          1              0.689908   
50   120000.0    -1         8          0              0.669504   
120   95000.0    -1        

In [2]:
import math
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

# =====================================================================
# 1. SAMRIDHI SCORE ENGINE MODEL
# =====================================================================

def evaluate_credit_score(features):
    """
    Simulates the alternate credit score computation logic from Samridhi.
    Input format: dict of user metrics.
    Output: final score (from 0 to 100).
    """
    baseline = 50
    score = baseline

    # 1. Profile type base adjustments
    role = features.get('profile_type', 'Guest')
    if role == 'Salaried':
        score += 15
    elif role == 'Freelancer':
        score += 8
    elif role == 'Entrepreneur':
        score += 12
    else:
        score -= 2

    # 2. Identity Verification
    if features.get('aadhaar_verified', False):
        score += 4
    if features.get('pan_verified', False):
        score += 3

    # 3. UPI Connectivity
    if features.get('upi_linked', False):
        score += 10
    if features.get('upi_verified', False):
        score += 15

    # 4. Skills Verification
    verified_skills = features.get('verified_skills_count', 0)
    score += verified_skills * 4

    # 5. Asset Ledgers
    if features.get('has_inventory', False):
        score += 10

    # 6. Biometrics & Documents
    if features.get('kyc_liveness_verified', False):
        score += 8
    if features.get('statement_ocr_uploaded', False):
        score += 7

    # 7. Portfolio Projects
    projects = features.get('verified_projects_count', 0)
    score += min(15, projects * 5)

    # 8. UPI Transaction Attributes
    tx_count = features.get('transaction_count', 0)
    score += min(8, int(tx_count * 0.5))

    surplus_ratio = features.get('surplus_inflow_ratio', 0.0) # (Inflows - Outflows) / Inflows
    if surplus_ratio > 0:
        score += min(12, int(surplus_ratio * 12))

    # 9. Deduct penalties for cashflow divergence outliers (anomalies)
    anomalous_tx_count = features.get('anomalous_transaction_count', 0)
    penalty = min(15, anomalous_tx_count * 5)
    score -= penalty

    # Keep credit score bound within [0, 100]
    return max(0, min(100, score))


# =====================================================================
# 2. CUSTOM SHAP FEATURE VALUE ATTRIBUTION
# =====================================================================

def calculate_shap_values(user_features):
    """
    Computes exact Shapley attribution values (contributions) for each feature.
    Baseline = 50. Output list of feature attributions summing to total score difference.
    """
    contributions = []

    # Profile Type Attribution
    role = user_features.get('profile_type', 'Guest')
    role_impact = 15 if role == 'Salaried' else 8 if role == 'Freelancer' else 12 if role == 'Entrepreneur' else -2
    contributions.append({'feature': 'Profile Type', 'value': role, 'impact': role_impact})

    # Aadhaar KYC
    if user_features.get('aadhaar_verified', False):
        contributions.append({'feature': 'Aadhaar Verified', 'value': True, 'impact': 4})

    # PAN KYC
    if user_features.get('pan_verified', False):
        contributions.append({'feature': 'PAN Verified', 'value': True, 'impact': 3})

    # UPI Link & Audit
    if user_features.get('upi_linked', False):
        contributions.append({'feature': 'UPI Connected', 'value': True, 'impact': 10})
    if user_features.get('upi_verified', False):
        contributions.append({'feature': 'UPI Log Audited', 'value': True, 'impact': 15})

    # Skills Verification
    skills = user_features.get('verified_skills_count', 0)
    if skills > 0:
        contributions.append({'feature': 'Skill Certifications', 'value': skills, 'impact': skills * 4})

    # Inventory Assets
    if user_features.get('has_inventory', False):
        contributions.append({'feature': 'Inventory Assets', 'value': True, 'impact': 10})

    # KYC Video Liveness
    if user_features.get('kyc_liveness_verified', False):
        contributions.append({'feature': 'KYC Video Liveness', 'value': True, 'impact': 8})

    # Statement OCR
    if user_features.get('statement_ocr_uploaded', False):
        contributions.append({'feature': 'Statement PDF OCR', 'value': True, 'impact': 7})

    # Verified Projects
    projects = user_features.get('verified_projects_count', 0)
    if projects > 0:
        contributions.append({'feature': 'Verified Projects', 'value': projects, 'impact': min(15, projects * 5)})

    # Transactions
    tx_count = user_features.get('transaction_count', 0)
    tx_vol = min(8, int(tx_count * 0.5))
    if tx_vol > 0:
        contributions.append({'feature': 'UPI Transaction Volume', 'value': tx_count, 'impact': tx_vol})

    surplus_ratio = user_features.get('surplus_inflow_ratio', 0.0)
    tx_surplus = min(12, int(surplus_ratio * 12)) if surplus_ratio > 0 else 0
    if tx_surplus > 0:
        contributions.append({'feature': 'UPI Surplus Cash Ratio', 'value': f"{surplus_ratio*100:.1f}%", 'impact': tx_surplus})

    # Divergence Outliers
    anom_count = user_features.get('anomalous_transaction_count', 0)
    penalty = min(15, anom_count * 5)
    if penalty > 0:
        contributions.append({'feature': 'Cashflow Outlier Penalties', 'value': anom_count, 'impact': -penalty})

    return contributions


# =====================================================================
# 3. LIME LOCAL SURROGATE MODEL
# =====================================================================

def calculate_lime_coefficients(user_features, num_perturbations=1000):
    """
    Generates local perturbations around the user's specific feature profile
    and fits an interpretable Ridge Regression model (Local Surrogate) to output
    LIME sensitivities.
    """
    feature_keys = [
        'aadhaar_verified', 'pan_verified', 'upi_linked', 'upi_verified',
        'has_inventory', 'kyc_liveness_verified', 'statement_ocr_uploaded',
        'anomaly_flag'
    ]

    # Original binary values
    x_original = np.array([
        1 if user_features.get('aadhaar_verified', False) else 0,
        1 if user_features.get('pan_verified', False) else 0,
        1 if user_features.get('upi_linked', False) else 0,
        1 if user_features.get('upi_verified', False) else 0,
        1 if user_features.get('has_inventory', False) else 0,
        1 if user_features.get('kyc_liveness_verified', False) else 0,
        1 if user_features.get('statement_ocr_uploaded', False) else 0,
        1 if user_features.get('anomalous_transaction_count', 0) > 0 else 0
    ])

    perturbations = []
    scores = []

    np.random.seed(42)
    for _ in range(num_perturbations):
        perturbed_x = x_original.copy()
        for idx in range(len(perturbed_x)):
            if np.random.rand() < 0.20:
                perturbed_x[idx] = 1 - perturbed_x[idx]

        perturbed_features = user_features.copy()
        perturbed_features.update({
            'aadhaar_verified': perturbed_x[0] == 1,
            'pan_verified': perturbed_x[1] == 1,
            'upi_linked': perturbed_x[2] == 1,
            'upi_verified': perturbed_x[3] == 1,
            'has_inventory': perturbed_x[4] == 1,
            'kyc_liveness_verified': perturbed_x[5] == 1,
            'statement_ocr_uploaded': perturbed_x[6] == 1,
            'anomalous_transaction_count': 2 if perturbed_x[7] == 1 else 0
        })

        score = evaluate_credit_score(perturbed_features)

        perturbations.append(perturbed_x)
        scores.append(score)

    X_perturb = np.array(perturbations)
    y_perturb = np.array(scores)

    # Compute Euclidean distances as weights to prioritize local neighbors
    distances = np.linalg.norm(X_perturb - x_original, axis=1)
    kernel_width = math.sqrt(len(feature_keys)) * 0.75
    weights = np.exp(- (distances ** 2) / (kernel_width ** 2))

    # Fit local Ridge regression surrogate
    model = Ridge(alpha=1.0)
    model.fit(X_perturb, y_perturb, sample_weight=weights)

    coefficients = {feature_keys[i]: float(model.coef_[i]) for i in range(len(feature_keys))}
    return model.intercept_, coefficients


# =====================================================================
# 4. RUN SUITE
# =====================================================================

if __name__ == "__main__":
    print("----------------------------------------------------------------------")
    print("      Samridhi: XAI (SHAP & LIME) Surrogate Training (Colab)         ")
    print("----------------------------------------------------------------------")

    # Freelancer profile example
    freelancer_profile = {
        'profile_type': 'Freelancer',          # Adds +8
        'aadhaar_verified': True,              # Adds +4
        'pan_verified': True,                  # Adds +3
        'upi_linked': True,                    # Adds +10
        'upi_verified': True,                  # Adds +15
        'verified_skills_count': 2,            # Adds 2 * 4 = +8
        'has_inventory': True,                 # Adds +10
        'kyc_liveness_verified': True,         # Adds +8
        'statement_ocr_uploaded': False,       # Adds 0
        'verified_projects_count': 1,          # Adds +5
        'transaction_count': 14,               # Adds +7
        'surplus_inflow_ratio': 0.45,          # Adds +5
        'anomalous_transaction_count': 1       # Deducts 1 * 5 = -5
    }

    # Evaluate Score
    final_score = evaluate_credit_score(freelancer_profile)
    print(f"User Alternate Credit Score: {final_score} / 100")

    # Run SHAP Attributions
    print("\n--- SHAP VALUES (FEATURE CONTRIBUTIONS) ---")
    contributions = calculate_shap_values(freelancer_profile)

    total_impact = 0
    for item in contributions:
        total_impact += item['impact']
        print(f"-> {item['feature']}: Impact = {item['impact']:+d} pts")
    print(f"Score Balance: Baseline (50) + Contributions ({total_impact:+d}) = {50 + total_impact}")

    # LIME Surrogate
    print("\n--- LIME LOCAL SURROGATE EXPLANATION ---")
    intercept, coeffs = calculate_lime_coefficients(freelancer_profile)

    terms = [f"{coef:+.2f} * {var}" for var, coef in coeffs.items()]
    print(f"y_approx ≈ {intercept:.2f} " + " ".join(terms))

----------------------------------------------------------------------
      Samridhi: XAI (SHAP & LIME) Surrogate Training (Colab)         
----------------------------------------------------------------------
User Alternate Credit Score: 100 / 100

--- SHAP VALUES (FEATURE CONTRIBUTIONS) ---
-> Profile Type: Impact = +8 pts
-> Aadhaar Verified: Impact = +4 pts
-> PAN Verified: Impact = +3 pts
-> UPI Connected: Impact = +10 pts
-> UPI Log Audited: Impact = +15 pts
-> Skill Certifications: Impact = +8 pts
-> Inventory Assets: Impact = +10 pts
-> KYC Video Liveness: Impact = +8 pts
-> Verified Projects: Impact = +5 pts
-> UPI Transaction Volume: Impact = +7 pts
-> UPI Surplus Cash Ratio: Impact = +5 pts
-> Cashflow Outlier Penalties: Impact = -5 pts
Score Balance: Baseline (50) + Contributions (+78) = 128

--- LIME LOCAL SURROGATE EXPLANATION ---
y_approx ≈ 96.73 +0.27 * aadhaar_verified +0.19 * pan_verified +0.86 * upi_linked +1.31 * upi_verified +0.75 * has_inventory +0.43 * kyc_live